In [3]:
import pandas as pd

df = pd.read_csv('/content/drive/MyDrive/Dataset/gym_dataset.csv')
df.head()

,text,label,label_id
0,What is today's class schedule?,class_schedule,0
1,Can you share the weekly class schedule?,class_schedule,0
2,What classes are available this week?,class_schedule,0
3,When is the next yoga class?,class_schedule,0
4,Is there a Zumba class tomorrow?,class_schedule,0


In [4]:
df['label_id'].unique()

array([0, 6, 4, 5, 7, 2, 3, 1])

In [5]:
from sklearn.preprocessing import LabelEncoder

# Initialize the encoder
le = LabelEncoder()

# Fit and transform the labels
df['labels_encoded'] = le.fit_transform(df['label'])

# Check the mapping
label_mapping = dict(zip(le.classes_, le.transform(le.classes_)))
print("Label mapping:", label_mapping)

# Inspect the new column
print(df[['label', 'labels_encoded']].head())

Label mapping: {'class_schedule': np.int64(0), 'contact_info': np.int64(1), 'facilities': np.int64(2), 'location': np.int64(3), 'membership_fee': np.int64(4), 'offers': np.int64(5), 'timings': np.int64(6), 'trainer_info': np.int64(7)}
            label  labels_encoded
0  class_schedule               0
1  class_schedule               0
2  class_schedule               0
3  class_schedule               0
4  class_schedule               0


In [6]:
from transformers import BertTokenizer

tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
print(tokenizer)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

BertTokenizer(name_or_path='bert-base-uncased', vocab_size=30522, model_max_length=512, padding_side='right', truncation_side='right', special_tokens={'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}, added_tokens_decoder={
	0: AddedToken("[PAD]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	100: AddedToken("[UNK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	101: AddedToken("[CLS]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	102: AddedToken("[SEP]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	103: AddedToken("[MASK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
}
)


In [7]:
text = df['text'].iloc[0]

token_ids = tokenizer.encode(
    text,
    max_length = 32,
    padding = 'max_length',
    truncation = True,
    return_tensors = 'pt')

print(token_ids)

tensor([[ 101, 2054, 2003, 2651, 1005, 1055, 2465, 6134, 1029,  102,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0]])


In [8]:
text = df['text'].iloc[0]

batch_encoder = tokenizer(
    text,
    max_length = 32,
    padding = 'max_length',
    truncation = True,
    return_tensors = 'pt')


print('Batch encoder keys:')
print(batch_encoder.keys())

print('nAttention mask:')
print(batch_encoder['attention_mask'])

Batch encoder keys:
KeysView({'input_ids': tensor([[ 101, 2054, 2003, 2651, 1005, 1055, 2465, 6134, 1029,  102,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0]])})
nAttention mask:
tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0]])


In [9]:
import torch

token_ids = []
attention_masks = []

for review in df['text']:           # ✅ iterate over reviews
    batch_encoder = tokenizer(
        review,                     # ✅ encode 'review', not 'text'
        max_length=32,              # ✅ smaller, consistent max_length
        padding='max_length',
        truncation=True,
        return_tensors='pt')

    token_ids.append(batch_encoder['input_ids'])
    attention_masks.append(batch_encoder['attention_mask'])

token_ids = torch.cat(token_ids, dim=0)
attention_masks = torch.cat(attention_masks, dim=0)

In [10]:
from sklearn.model_selection import train_test_split
from torch.utils.data import TensorDataset, DataLoader

label_id = torch.tensor(df['labels_encoded'].values)

# Split everything together using indices
train_ids, val_ids, train_masks, val_masks, train_labels, val_labels = train_test_split(
    token_ids, attention_masks, label_id,
    test_size=0.1,
    shuffle=True,
    random_state=42
)

train_data = TensorDataset(train_ids, train_masks, train_labels)
train_dataloader = DataLoader(train_data, shuffle=True, batch_size=8)
val_data = TensorDataset(val_ids, val_masks, val_labels)
val_dataloader = DataLoader(val_data, batch_size=16)

In [11]:
from transformers import BertForSequenceClassification

model = BertForSequenceClassification.from_pretrained(
    'bert-base-uncased',
    num_labels=8)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [12]:
from torch.optim import AdamW
import torch.nn as nn
from transformers import get_linear_schedule_with_warmup

EPOCHS = 20

# Optimizer
optimizer = AdamW(model.parameters(), lr=1e-5, weight_decay=0.01)
# Loss function
loss_function = nn.CrossEntropyLoss()

# Scheduler
num_training_steps = EPOCHS * len(train_dataloader)

num_warmup_steps = int(0.1 * num_training_steps)  # 10% warmup

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=num_warmup_steps,
    num_training_steps=num_training_steps
)

In [13]:
def calculate_accuracy(preds, labels):
    """ Calculate the accuracy of model predictions against true labels.

    Parameters:
        preds (np.array): The predicted label from the model
        labels (np.array): The true label

    Returns:
        accuracy (float): The accuracy as a percentage of the correct
            predictions.
    """
    pred_flat = np.argmax(preds, axis=1).flatten()
    labels_flat = labels.flatten()
    accuracy = np.sum(pred_flat == labels_flat) / len(labels_flat)

    return accuracy

In [14]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

for epoch in range(EPOCHS):

    model.train()
    training_loss = 0

    for batch in train_dataloader:

        batch_token_ids = batch[0].to(device)
        batch_attention_mask = batch[1].to(device)
        batch_labels = batch[2].to(device)

        model.zero_grad()

        loss, logits = model(
            batch_token_ids,
            token_type_ids=None,
            attention_mask=batch_attention_mask,
            labels=batch_labels,
            return_dict=False
        )

        training_loss += loss.item()

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

        optimizer.step()
        scheduler.step()

    average_train_loss = training_loss / len(train_dataloader)
    print(f"Epoch {epoch+1} | Avg Train Loss: {average_train_loss:.4f}")

Epoch 1 | Avg Train Loss: 2.0924
Epoch 2 | Avg Train Loss: 2.0665
Epoch 3 | Avg Train Loss: 1.8595
Epoch 4 | Avg Train Loss: 1.7111
Epoch 5 | Avg Train Loss: 1.5799
Epoch 6 | Avg Train Loss: 1.4532
Epoch 7 | Avg Train Loss: 1.2901
Epoch 8 | Avg Train Loss: 1.1540
Epoch 9 | Avg Train Loss: 1.0136
Epoch 10 | Avg Train Loss: 0.9025
Epoch 11 | Avg Train Loss: 0.7719
Epoch 12 | Avg Train Loss: 0.6889
Epoch 13 | Avg Train Loss: 0.5897
Epoch 14 | Avg Train Loss: 0.5497
Epoch 15 | Avg Train Loss: 0.4852
Epoch 16 | Avg Train Loss: 0.4322
Epoch 17 | Avg Train Loss: 0.4140
Epoch 18 | Avg Train Loss: 0.3841
Epoch 19 | Avg Train Loss: 0.3740
Epoch 20 | Avg Train Loss: 0.3642


In [15]:
import numpy as np
model.eval()
val_loss = 0
val_accuracy = 0

for batch in val_dataloader:

    batch_token_ids = batch[0].to(device)
    batch_attention_mask = batch[1].to(device)
    batch_labels = batch[2].to(device)

    with torch.no_grad():
        (loss, logits) = model(
            batch_token_ids,
            attention_mask = batch_attention_mask,
            labels = batch_labels,
            token_type_ids = None,
            return_dict=False)

    logits = logits.detach().cpu().numpy()
    label_ids = batch_labels.to('cpu').numpy()
    val_loss += loss.item()
    val_accuracy += calculate_accuracy(logits, label_ids)

average_val_accuracy = val_accuracy / len(val_dataloader)

In [16]:
print(average_val_accuracy)

0.875


In [17]:
from google.colab import files
model_path = "New_Bert_model.pth"
torch.save(model.state_dict(), model_path)
print(f"Model saved as {model_path}")

files.download("New_Bert_model.pth")

Model saved as New_Bert_model.pth


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [18]:
# Load tokenizer
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

# Initialize model
model = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=8  # make sure this matches your trained model
)

# Load your trained weights
model.load_state_dict(torch.load("New_Bert_model.pth", map_location="cpu"))

# Put model in evaluation mode
model.eval()

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12,

In [19]:
text = "what are the gym hours?	"

inputs = tokenizer(
    text,
    max_length=32,
    padding='max_length',
    truncation=True,
    return_tensors="pt"
)

In [20]:
with torch.no_grad():
    outputs = model(
        input_ids=inputs['input_ids'],
        attention_mask=inputs['attention_mask']
    )
    logits = outputs.logits

# Get predicted class
predicted_class_id = torch.argmax(logits, dim=1).item()
print("Predicted class ID:", predicted_class_id)

Predicted class ID: 6


In [21]:
label_mapping = dict(df[['label', 'labels_encoded']].drop_duplicates().values)
print(label_mapping)

{'class_schedule': 0, 'timings': 6, 'membership_fee': 4, 'offers': 5, 'trainer_info': 7, 'facilities': 2, 'location': 3, 'contact_info': 1}
